# Why One Model Is Never Enough
### MSc Data Science for Business — Week 10 Workshop

---

**Business scenario:** You are a data scientist working for a hotel group. Management want to predict which bookings are likely to be cancelled so they can make smarter decisions about overbooking policy, deposit requirements, and targeted retention offers.

**Today's question:** Does it matter which model we choose?

**Our journey:**
1. Load and explore the data
2. Engineer features and encode categoricals
3. Train a Logistic Regression baseline
4. Train a single Decision Tree
5. Train a Random Forest ensemble
6. Compare all three and discuss

> 💬 **Before we write any code:** Based on what you already know about logistic regression and decision trees — how well do you think each will perform? Which do you expect to win? We'll come back to your predictions at the end.


---
## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42

print('Libraries loaded.')

In [ ]:
df = pd.read_csv('hotel_bookings_clean.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nCancellation rate: {df["is_canceled"].mean():.1%}')
df.head()

---
## 2. Quick Exploration

Before modelling, always understand what you're working with.

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cancellation overall
df['is_canceled'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Cancellations Overall')
axes[0].set_xticklabels(['Not Cancelled', 'Cancelled'], rotation=0)
axes[0].set_ylabel('Count')

# Cancellation rate by market segment
cancel_by_segment = df.groupby('market_segment')['is_canceled'].mean().sort_values()
colours = ['tomato' if v > df['is_canceled'].mean() else 'steelblue' for v in cancel_by_segment]
cancel_by_segment.plot(kind='bar', ax=axes[1], color=colours)
axes[1].axhline(df['is_canceled'].mean(), color='black', linewidth=1, linestyle='--', label='Overall average')
axes[1].set_title('Cancellation Rate by Market Segment')
axes[1].set_ylabel('Cancellation Rate')
axes[1].set_xticklabels(cancel_by_segment.index, rotation=30, ha='right')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Lead time distribution by cancellation status
fig, ax = plt.subplots(figsize=(10, 4))

for label, colour in [(0, 'steelblue'), (1, 'tomato')]:
    subset = df[df['is_canceled'] == label]['lead_time']
    ax.hist(subset, bins=60, alpha=0.5, color=colour,
            label='Not Cancelled' if label == 0 else 'Cancelled', density=True)

ax.set_xlabel('Lead Time (days)')
ax.set_ylabel('Density')
ax.set_title('Lead Time Distribution by Cancellation Status')
ax.legend()
plt.tight_layout()
plt.show()

> 💬 **Discussion:** What do you notice about lead time and cancellation? Does this match your intuition about how people book hotels?

---
## 3. Feature Engineering

We need to make deliberate decisions about which columns to include and how to prepare them. Every drop is a judgement call — document your reasoning.

**But first — a key point worth stating clearly:**

Every model in this notebook is doing mathematics. Logistic regression, decision trees, random forests — they all ultimately operate on numbers. In logistic regression, the prediction equation is computed across thousands of rows via matrix multiplication — a string like `"Online TA"` cannot be multiplied by a coefficient. So regardless of which tool you use — JMP, R, scikit-learn — **categories always have to become numbers before a model can use them.**

JMP does this silently. We're doing it explicitly, so you can see exactly what's happening.

> **Note on terminology:** If you've used JMP's decision trees, you'll have seen "train" and "validate" sets. Here we use "train" and "test" — same concept, different name. We'll keep things to two sets for now and park the full train/validate/test split for another time.


In [ ]:
# --- Columns to drop and why ---
cols_to_drop = [
    'arrival_date_year',       # Only 2015–2017; not generalisable to future bookings
    'arrival_date_week_number',# Cyclical — would need sine/cosine encoding to use properly
    'arrival_date_day_of_month',# Same issue; adds noise without clear payoff
    'reserved_room_type',      # Opaque letter codes; low interpretability
    'assigned_room_type',      # Same
]

df_model = df.drop(columns=cols_to_drop).copy()
print(f'Columns remaining: {df_model.shape[1]}')

---
### 📖 Concept: Frequency Encoding

Some categorical columns have too many unique values to one-hot encode sensibly. `agent` has 334 unique values and `country` has 178 — one-hot encoding those would add 500+ mostly-empty columns to our feature matrix.

**Frequency encoding** replaces each category with its proportion in the dataset:

| agent | → | agent (frequency encoded) |
|-------|---|--------------------------|
| 9     | → | 0.031 (appears in 3.1% of rows) |
| 42    | → | 0.008 (appears in 0.8% of rows) |

The intuition is that how *common* an agent or country is carries real signal — a dominant agent handling 20% of bookings probably behaves differently from a tiny one handling 0.1%. Frequency becomes a proxy for that.

**Trade-off:** Two agents with identical booking volumes become numerically indistinguishable, even if they behave differently. It's a lossy encoding — we're sacrificing identity for dimensionality. For `agent` and `country`, that's an acceptable compromise.


In [ ]:
# --- Frequency encode high-cardinality categoricals ---
# 'agent' has 334 unique values; 'country' has 178.
# One-hot encoding would create hundreds of near-empty columns.
# Instead we replace each value with its frequency in the dataset.

# Capture before snapshot
before = df_model[['agent', 'country']].head(10).copy()

for col in ['agent', 'country']:
    freq = df_model[col].value_counts(normalize=True)
    df_model[col] = df_model[col].map(freq)

print('Before encoding:')
display(before)
print('\nAfter frequency encoding:')
display(df_model[['agent', 'country']].head(10))


In [ ]:
# --- One-hot encode low-cardinality categoricals ---
# drop_first=True drops one category per variable (the reference category)
# to avoid multicollinearity

low_card_cats = [
    'hotel', 'arrival_date_month', 'meal',
    'market_segment', 'distribution_channel',
    'deposit_type', 'customer_type'
]

df_model = pd.get_dummies(df_model, columns=low_card_cats, drop_first=True)

# Cast booleans to int — newer pandas returns bool from get_dummies,
# which can cause issues in StandardScaler
df_model = df_model.apply(lambda x: x.astype(int) if x.dtype == bool else x)

print(f'Final feature matrix shape: {df_model.shape}')
print(f'\nSample of generated one-hot columns for meal and deposit_type:')
meal_cols = [c for c in df_model.columns if c.startswith('meal_')]
deposit_cols = [c for c in df_model.columns if c.startswith('deposit_')]
display(df_model[meal_cols + deposit_cols].head(8))


---
### 📖 Concept: One-Hot Encoding

Machine learning models need numbers. But if we encode `meal` as BB=1, HB=2, FB=3, SC=4, the model thinks HB is mathematically "twice" BB — which is meaningless. There's no numeric relationship between meal plan categories.

**One-hot encoding** creates a new binary column for each category:

| meal_HB | meal_FB | meal_SC |
|---------|---------|---------|
| 1       | 0       | 0       |
| 0       | 1       | 0       |
| 0       | 0       | 1       |

Each row gets a 1 in exactly one column — no false numeric relationship implied.

We use `drop_first=True` to drop one column per variable (the reference category). With 4 meal types, we only need 3 binary columns — the 4th is implied when all others are 0. This avoids **multicollinearity**, where one column can be perfectly predicted from the others.

> **Note:** Random Forest doesn't mathematically *need* one-hot encoding — trees split on equality, not magnitude. But scikit-learn requires numeric input, so we encode regardless.


---
## 4. Train / Test Split

In [ ]:
X = df_model.drop(columns=['is_canceled'])
y = df_model['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set:  {X_train.shape[0]:,} rows')
print(f'Test set:      {X_test.shape[0]:,} rows')
print(f'\nCancellation rate — train: {y_train.mean():.1%} | test: {y_test.mean():.1%}')

In [ ]:
# Scale features for logistic regression
# Tree-based models don't need scaling — but we create a scaled version for LR

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# We fit the scaler on training data only, then apply the same transformation
# to the test set. If we scaled on the full dataset first, the scaler would
# incorporate test set values into the mean and standard deviation — a subtle
# form of data leakage. The test set must remain a sealed envelope throughout.
# In practice on a large dataset the difference is small, but the principle
# matters: in production, future data doesn't exist yet when you train.
print('Scaling complete.')


---
### 📖 Concept: Train / Test Split

We divide our data into two sets:

- **Training set (80%)** — the model learns from this data
- **Test set (20%)** — held back completely until evaluation

The test set simulates data the model has never seen, giving us an honest estimate of real-world performance. If we evaluated on the training data, we'd be measuring memorisation, not learning.

`stratify=y` ensures both sets have the same cancellation rate — important when the target classes aren't perfectly balanced.


> 💬 **Discussion:** Why do we fit the scaler on the training set only, and then apply it to the test set? What would go wrong if we scaled on the whole dataset first?

---
## 5. Model 1 — Logistic Regression

Our baseline. You already know this model — let's see how far it gets us.

---
### 📖 Concept: Feature Scaling

Logistic regression uses gradient descent to find the best coefficients. If one feature ranges from 0–500 (like `lead_time`) and another from 0–1 (like `is_repeated_guest`), the algorithm treats the large-scale feature as disproportionately important.

`StandardScaler` fixes this by transforming each feature to have mean = 0 and standard deviation = 1.

**Why fit on training data only?** If we scaled on the full dataset first, information from the test set would leak into the training process — the scaler would "know" the test set's range and mean. We fit on training, then apply the same transformation to the test set.

> Tree-based models (Decision Tree, Random Forest) are scale-invariant — they split on thresholds, not magnitudes. We use unscaled data for those.


In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Not Cancelled', 'Cancelled']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_lr):.4f}')

In [ ]:
# Confusion matrix — raw counts and normalised side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    display_labels=['Not Cancelled', 'Cancelled'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Logistic Regression — Counts')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    display_labels=['Not Cancelled', 'Cancelled'],
    cmap='Blues', ax=axes[1], normalize='true'
)
axes[1].set_title('Logistic Regression — Normalised (% of actual)')

plt.tight_layout()
plt.show()


In [ ]:
# Top coefficients — what does LR think matters?
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
colours = ['tomato' if c > 0 else 'steelblue' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colours)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression — Top 15 Coefficients')
ax.set_xlabel('Coefficient (positive = more likely to cancel)')
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Look at the top coefficients. Do they make business sense? Are there any surprises? What does a negative coefficient mean here?

---
## 6. Model 2 — Decision Tree

You've seen decision trees before. Let's see what happens when we let one grow freely — and then when we constrain it.

In [ ]:
# Unconstrained tree — what happens?
dt_full = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_full.fit(X_train, y_train)

y_pred_dt_full = dt_full.predict(X_test)
y_prob_dt_full = dt_full.predict_proba(X_test)[:, 1]

print('=== Decision Tree (unconstrained) ===')
print(f'Tree depth: {dt_full.get_depth()}')
print(f'Number of leaves: {dt_full.get_n_leaves()}')
print(classification_report(y_test, y_pred_dt_full, target_names=['Not Cancelled', 'Cancelled']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_dt_full):.4f}')

> 💬 **Discussion:** The unconstrained tree has memorised the training data. What is this called? How does the tree depth relate to this problem?

---
### 📖 Concept: The Confusion Matrix

A confusion matrix shows the four possible outcomes of a binary prediction:

|                        | **Predicted: Not Cancelled** | **Predicted: Cancelled** |
|------------------------|------------------------------|--------------------------|
| **Actually: Not Cancelled** | True Negative (TN) ✅   | False Positive (FP) ❌   |
| **Actually: Cancelled**     | False Negative (FN) ❌  | True Positive (TP) ✅    |

- **False Positive** — we predicted cancellation, but the guest stayed. We may have unnecessarily offered a retention discount.
- **False Negative** — we missed a cancellation. The room went unprotected; no overbooking buffer, no targeted offer.

Are these costs symmetric? That's a business question, not a statistics question.


In [ ]:
# Constrained tree — more generalisable
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print('=== Decision Tree (max_depth=5) ===')
print(classification_report(y_test, y_pred_dt, target_names=['Not Cancelled', 'Cancelled']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_dt):.4f}')

In [ ]:
# Visualise the top of the tree
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt, max_depth=3,
    feature_names=X.columns,
    class_names=['Not Cancelled', 'Cancelled'],
    filled=True, rounded=True, fontsize=9, ax=ax
)
ax.set_title('Decision Tree — First 3 Levels')
plt.tight_layout()
plt.show()

> 💬 **Discussion:** A non-technical hotel manager could (just about) follow this tree. Is that an advantage? When would interpretability matter more than performance?

---
## 7. Model 3 — Random Forest

Instead of one tree, we build many — each trained on a random sample of the data and a random subset of features. The ensemble votes.

**Key intuition:** One opinionated expert can be wrong. Many slightly different experts, averaged, are more robust. This is *bagging* — Bootstrap AGGregating.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=20,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest (200 trees) ===')
print(classification_report(y_test, y_pred_rf, target_names=['Not Cancelled', 'Cancelled']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}')


---
### 📖 Concept: Feature Importance

Random Forests can tell us which features were most useful across all 200 trees. **Mean Decrease in Impurity** measures how much each feature reduced uncertainty (Gini impurity) at the splits where it was used, averaged across all trees.

A high importance score means the feature was frequently chosen for splits and produced clean separations between cancellations and non-cancellations.

**Caveat:** This measure can overstate the importance of high-cardinality numeric features. It's a useful guide, not gospel — always sense-check against business logic.


In [ ]:
# Feature importance — what does the forest think matters?
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance_df['feature'][::-1], importance_df['importance'][::-1], color='steelblue')
ax.set_title('Random Forest — Top 15 Feature Importances')
ax.set_xlabel('Mean Decrease in Impurity')
plt.tight_layout()
plt.show()

> 💬 **Discussion:** Compare this feature importance chart to the logistic regression coefficients. Do the models agree on what matters? Where do they disagree — and why might that be?

---
## 8. Comparing All Three Models

---
### 📖 Concept: The ROC Curve and AUC

Every classifier has a **threshold** — the probability above which it predicts "cancelled." Lowering the threshold catches more cancellations (higher True Positive Rate) but also flags more non-cancellations as cancelled (higher False Positive Rate).

The **ROC curve** plots TPR vs FPR across every possible threshold from 1.0 down to 0.0:

- Bottom-left (threshold = 1.0): model flags nothing — TPR = 0, FPR = 0
- Top-right (threshold = 0.0): model flags everything — TPR = 1, FPR = 1  
- **Ideal model:** shoots straight up to TPR = 1.0 before FPR moves at all — hugging the top-left corner
- **Random classifier:** diagonal line — catches true positives and false positives at the same rate

**AUC (Area Under the Curve)** summarises the whole curve as a single number. A random classifier scores 0.5; a perfect model scores 1.0. AUC is particularly useful because it's threshold-independent — it tells you about the model's overall discriminative ability regardless of where you set the cutoff.


In [ ]:
# ROC curves — all three on one chart
fig, ax = plt.subplots(figsize=(8, 6))

models = [
    ('Logistic Regression', y_prob_lr, 'steelblue'),
    ('Decision Tree',       y_prob_dt, 'tomato'),
    ('Random Forest',       y_prob_rf, 'seagreen'),
]

for name, probs, colour in models:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=colour, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []
for name, y_pred, y_prob in [
    ('Logistic Regression', y_pred_lr, y_prob_lr),
    ('Decision Tree',       y_pred_dt, y_prob_dt),
    ('Random Forest',       y_pred_rf, y_prob_rf),
]:
    results.append({
        'Model':     name,
        'Accuracy':  f'{accuracy_score(y_test, y_pred):.3f}',
        'Precision': f'{precision_score(y_test, y_pred):.3f}',
        'Recall':    f'{recall_score(y_test, y_pred):.3f}',
        'F1':        f'{f1_score(y_test, y_pred):.3f}',
        'ROC-AUC':   f'{roc_auc_score(y_test, y_prob):.3f}',
    })

pd.DataFrame(results).set_index('Model')


---
### 📖 Reading the Results Table

Here's what each column is actually telling you:

**Accuracy** — of all predictions made, what proportion were correct? Random Forest gets 85.6% right overall. Sounds impressive, but accuracy alone can be misleading — a model that predicted "not cancelled" for every single booking would still be right 63% of the time.

**Precision** — of all the bookings we *predicted* would cancel, what proportion actually did? Random Forest: 87.5% — when it says cancel, it's right most of the time. The Decision Tree is the weakest here at 69.3%, generating the most false alarms.

**Recall** — of all the bookings that *actually* cancelled, what proportion did we catch? This is where it gets interesting. The Decision Tree catches the most at 77.6%; the Random Forest gets 71.4%; Logistic Regression only catches 64.3% — missing more than a third of real cancellations.

**F1** — the harmonic mean of precision and recall. Unlike a regular average, it penalises large gaps between the two — you can't score well by being brilliant at one and terrible at the other. Random Forest wins at 0.786.

**ROC-AUC** — overall discriminative ability across all possible thresholds, regardless of where you set the decision boundary. Random Forest wins comfortably at 0.936.

**The tension in this table:** the Decision Tree has the best recall but the worst precision. Logistic Regression has decent precision but the worst recall. Random Forest wins on almost everything — except recall, where the Decision Tree edges it.

**The business question this raises:** for a hotel, is it worse to *miss* a cancellation (false negative — room sits empty, revenue lost) or to *wrongly predict* one (false positive — unnecessary discount offered, or room sold twice)? The answer determines which metric to prioritise — and that's a business decision, not a statistical one.


---
## 9. The Business Conversation

You now have three trained models and a comparison table. But the job isn't done.

**Discuss in your groups:**

1. **Which model would you deploy — and why?** Think beyond the metrics. Consider interpretability, maintenance, stakeholder trust.

2. **Precision vs Recall — which matters more here?** If the model predicts a cancellation and it doesn't cancel, what happens? If it misses a cancellation, what happens? Are the costs symmetric?

3. **What could go wrong in production?** How would you know if the model was degrading over time? What might cause it to degrade?

4. **How would you explain the Random Forest's decision to a hotel manager** who has never heard of machine learning?

---
## 10. Extension — How Many Trees Is Enough?

*If time allows — or explore independently after the session.*

In [ ]:
# How does AUC change as we add more trees?
tree_counts = [1, 5, 10, 25, 50, 100, 200, 300, 500]
auc_scores = []

for n in tree_counts:
    rf_n = RandomForestClassifier(
        n_estimators=n, max_depth=15,
        min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_n.fit(X_train, y_train)
    auc_scores.append(roc_auc_score(y_test, rf_n.predict_proba(X_test)[:, 1]))
    print(f'  {n:>4} trees → AUC: {auc_scores[-1]:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tree_counts, auc_scores, marker='o', color='steelblue', linewidth=2)
ax.set_xlabel('Number of Trees')
ax.set_ylabel('ROC-AUC')
ax.set_title('Random Forest — Performance vs Number of Trees')
plt.tight_layout()
plt.show()

> 💬 **Discussion:** At what point does adding more trees stop helping? What does this tell you about the relationship between model complexity and performance — and about the cost of computation?

---
*MSc Data Science for Business — Regent's University London*